# Preprocessing

All data preparation for the project, in pipeline order:

1. **Setup**: paths, districts, and which steps to run
2. **NASA POWER district weather**: daily, 11 variables, 2000 to today (this is what produced `data/weather/v2`)
3. **Regional upstream points + climate indices**: extra predictors (MJO, ENSO, IOD)
4. **Open-Meteo forecast**: live 16-day NWP forecast
5. **Cleaning**: gap-filling of the training data, written to `data/weather/clean/` for `forecasting/lstm.ipynb`

Steps 2-4 download from the internet and are switched off by default (flags in the setup cell), so *Run All* only re-runs the cleaning.

## 1. Setup

In [ ]:
import time
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import requests

# Repo root = the folder that contains both codes/ and data/ (works on any machine)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "codes").is_dir() and (p / "data").is_dir())
WEATHER_DIR = ROOT / "data" / "weather"

DISTRICTS = {
    "Beed":       {"latitude": 18.9891, "longitude": 75.7601},
    "Latur":      {"latitude": 18.4088, "longitude": 76.5604},
    "Osmanabad":  {"latitude": 18.1860, "longitude": 76.0419},
    "Solapur":    {"latitude": 17.6599, "longitude": 75.9064},
    "Ahmednagar": {"latitude": 19.0948, "longitude": 74.7480},
}
START_DATE = date(2000, 1, 1)
END_DATE = date.today()

# ---- Which steps to run ----
RUN_NASA_DOWNLOAD = False      # step 2 -> data/weather/<DOWNLOAD_VERSION>/
RUN_EXTENDED_DOWNLOAD = False  # step 3 -> data/weather/extended/
RUN_FORECAST_DOWNLOAD = False  # step 4 -> data/weather/forecast/
RUN_CLEANING = True            # step 5 -> data/weather/clean/

# A fresh download goes to a NEW version folder so it never overwrites v2,
# the exact data the current LSTM models were trained on.
DOWNLOAD_VERSION = "v3"
# The version that gets cleaned for the LSTM (change to "v3" to train on new data)
TRAINING_VERSION = "v2"
TRAIN_END = "2018-12-31"       # must match TRAIN_END in lstm.ipynb (cloud-cover climatology uses train years only)

print("repo root:", ROOT)

## 2. NASA POWER district weather

Daily point data for the 5 districts. The 11 variables are the original 8 plus three added for forecasting:
dew point (moisture), cloud amount (near-term rain likelihood) and 50 m wind (larger-scale flow).
NASA POWER marks missing values with -999; those become NaN.

In [ ]:
NASA_URL = "https://power.larc.nasa.gov/api/temporal/daily/point"
NASA_PARAMS = {
    "PRECTOTCORR": "Rainfall_mm",
    "T2M": "Temperature_C",
    "T2M_MAX": "Temperature_Max_C",
    "T2M_MIN": "Temperature_Min_C",
    "RH2M": "Relative_Humidity_percent",
    "WS10M": "Wind_Speed_mps",
    "PS": "Surface_Pressure_kPa",
    "ALLSKY_SFC_SW_DWN": "Solar_Radiation",
    "T2MDEW": "Dew_Point_C",
    "CLOUD_AMT": "Cloud_Amount_percent",
    "WS50M": "Wind_Speed_50m_mps",
}


def download_nasa_district(coords):
    params = {
        "parameters": ",".join(NASA_PARAMS),
        "community": "AG",
        "longitude": coords["longitude"],
        "latitude": coords["latitude"],
        "start": START_DATE.strftime("%Y%m%d"),
        "end": END_DATE.strftime("%Y%m%d"),
        "format": "JSON",
        "time-standard": "LST",
    }
    response = requests.get(NASA_URL, params=params, timeout=60)
    response.raise_for_status()
    weather = pd.DataFrame(response.json()["properties"]["parameter"])
    weather.index.name = "DATE"
    weather.reset_index(inplace=True)
    weather["DATE"] = pd.to_datetime(weather["DATE"], format="%Y%m%d")
    weather.replace(-999, pd.NA, inplace=True)
    return weather.rename(columns=NASA_PARAMS)


if RUN_NASA_DOWNLOAD:
    out_dir = WEATHER_DIR / DOWNLOAD_VERSION
    out_dir.mkdir(parents=True, exist_ok=True)
    for name, coords in DISTRICTS.items():
        weather = download_nasa_district(coords)
        out = out_dir / f"{name.lower()}_weather.csv"
        weather.to_csv(out, index=False)
        print(f"{name}: {len(weather)} rows, {int(weather.isna().sum().sum())} missing -> {out}")
else:
    print("skipped (RUN_NASA_DOWNLOAD = False)")

## 3. Regional upstream points + climate indices

Monsoon rain systems usually form over the Bay of Bengal and travel west-northwest across central India,
so weather at eastern points shows up in these districts a few days later; western points track the
moisture-carrying winds from the Arabian Sea. Climate indices: **MJO** (daily, active/break monsoon phases),
**ENSO/ONI** and **IOD/DMI** (monthly background state).

District files here carry 14 variables (the 11 above + wind direction x2 + specific humidity), so they are
*not* drop-in replacements for `v2`. Re-runs are incremental (only the last ~60 days are re-downloaded).

When building features, use the *previous* month's ONI/DMI (they are published after the month ends).

In [ ]:
EXT_DIR = WEATHER_DIR / "extended"
OUT_DIR = EXT_DIR
DISTRICT_DIR = EXT_DIR / "districts"
REGIONAL_DIR = EXT_DIR / "regional"
INDEX_DIR = EXT_DIR / "indices"
REFRESH_DAYS = 60  # re-download this many recent days on every run

# ============================================
# Regional upstream points (predictors only)
# The 5 districts are close together, so one shared set serves all of them.
# ============================================
REGIONAL_POINTS = {
    # East / south-east: path of Bay of Bengal monsoon lows
    "Nanded":             {"latitude": 19.1383, "longitude": 77.3210, "direction": "east"},
    "Hyderabad":          {"latitude": 17.3850, "longitude": 78.4867, "direction": "southeast"},
    "Nagpur":             {"latitude": 21.1458, "longitude": 79.0882, "direction": "northeast"},
    "Raipur":             {"latitude": 21.2514, "longitude": 81.6296, "direction": "east"},
    "Visakhapatnam":      {"latitude": 17.6868, "longitude": 83.2185, "direction": "east"},
    "Bhubaneswar":        {"latitude": 20.2961, "longitude": 85.8245, "direction": "east"},
    "BayOfBengal_North":  {"latitude": 19.0000, "longitude": 88.0000, "direction": "east"},   # ocean point
    # West: monsoon winds and moisture from the Arabian Sea
    "Pune":               {"latitude": 18.5204, "longitude": 73.8567, "direction": "west"},
    "Mumbai":             {"latitude": 19.0760, "longitude": 72.8777, "direction": "west"},
    "ArabianSea_East":    {"latitude": 18.0000, "longitude": 71.0000, "direction": "west"},   # ocean point
}

# ============================================
# NASA POWER parameters
# Original 8 + your 3 additions kept. Added:
#   WD10M : wind direction @10m -- monsoon westerlies vs. break-phase shifts
#   WD50M : wind direction @50m -- larger-scale flow direction
#   QV2M  : specific humidity   -- absolute moisture, unlike RH it doesn't
#           swing with temperature during the day
# Regional points use a smaller set: only what helps predict rain arriving.
# ============================================
DISTRICT_PARAMS = {
    "PRECTOTCORR": "Rainfall_mm",
    "T2M": "Temperature_C",
    "T2M_MAX": "Temperature_Max_C",
    "T2M_MIN": "Temperature_Min_C",
    "RH2M": "Relative_Humidity_percent",
    "WS10M": "Wind_Speed_mps",
    "PS": "Surface_Pressure_kPa",
    "ALLSKY_SFC_SW_DWN": "Solar_Radiation",
    "T2MDEW": "Dew_Point_C",
    "CLOUD_AMT": "Cloud_Amount_percent",
    "WS50M": "Wind_Speed_50m_mps",
    "WD10M": "Wind_Direction_deg",
    "WD50M": "Wind_Direction_50m_deg",
    "QV2M": "Specific_Humidity_gkg",
}

REGIONAL_PARAMS = {
    "PRECTOTCORR": "Rainfall_mm",
    "T2M_MAX": "Temperature_Max_C",
    "RH2M": "Relative_Humidity_percent",
    "PS": "Surface_Pressure_kPa",
    "T2MDEW": "Dew_Point_C",
    "QV2M": "Specific_Humidity_gkg",
    "CLOUD_AMT": "Cloud_Amount_percent",
    "WS50M": "Wind_Speed_50m_mps",
    "WD50M": "Wind_Direction_50m_deg",
}

POWER_URL = "https://power.larc.nasa.gov/api/temporal/daily/point"


# ============================================
# Helpers
# ============================================
def get_with_retry(url, params=None, timeout=120, retries=4):
    """GET with exponential backoff (NASA POWER rate-limits and occasionally times out)."""
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            if r.status_code == 422:
                # Invalid request, usually a parameter name -- retrying won't help
                raise ValueError(f"Request rejected (422): {r.text[:500]}")
            r.raise_for_status()
            return r
        except (requests.RequestException,) as e:
            if attempt == retries:
                raise
            wait = 5 * 2 ** (attempt - 1)
            print(f"  attempt {attempt} failed ({e}); retrying in {wait}s")
            time.sleep(wait)


def fetch_power(lat, lon, param_map, start, end):
    """Download one point from NASA POWER and return a tidy daily DataFrame."""
    params = {
        "parameters": ",".join(param_map),
        "community": "AG",
        "latitude": lat,
        "longitude": lon,
        "start": start.strftime("%Y%m%d"),
        "end": end.strftime("%Y%m%d"),
        "format": "JSON",
        "time-standard": "LST",
    }
    data = get_with_retry(POWER_URL, params).json()

    df = pd.DataFrame(data["properties"]["parameter"])
    df.index = pd.to_datetime(df.index, format="%Y%m%d")
    df.index.name = "DATE"
    df = df.reset_index()

    # -999 is NASA POWER's missing-value sentinel
    df = df.replace(-999, pd.NA)
    df = df.rename(columns=param_map)

    # Drop trailing days that NASA POWER hasn't processed yet (all values missing)
    value_cols = [c for c in df.columns if c != "DATE"]
    df = df.dropna(subset=value_cols, how="all")
    return df


def update_point(name, coords, param_map, out_dir):
    """Create the file, or incrementally update it if it already exists."""
    path = out_dir / f"{name.lower()}_weather.csv"
    old = None
    start = START_DATE

    if path.exists():
        old = pd.read_csv(path, parse_dates=["DATE"])
        missing_cols = set(param_map.values()) - set(old.columns)
        if missing_cols:
            print(f"  new columns {sorted(missing_cols)} -> full re-download")
            old = None
        else:
            start = max(START_DATE, old["DATE"].max().date() - timedelta(days=REFRESH_DAYS))

    print(f"  downloading {start} -> {END_DATE}")
    new = fetch_power(coords["latitude"], coords["longitude"], param_map, start, END_DATE)

    if old is not None:
        # New download wins for overlapping dates (replaces provisional values)
        combined = pd.concat([old[old["DATE"] < pd.Timestamp(start)], new], ignore_index=True)
    else:
        combined = new

    combined = combined.drop_duplicates("DATE", keep="last").sort_values("DATE")
    combined.to_csv(path, index=False)

    # Quick data-quality report
    expected = pd.date_range(combined["DATE"].min(), combined["DATE"].max(), freq="D")
    gaps = len(expected) - len(combined)
    n_missing = int(combined.drop(columns="DATE").isna().sum().sum())
    print(f"  saved {path}  rows={len(combined)}  last={combined['DATE'].max().date()}  "
          f"missing_dates={gaps}  missing_values={n_missing}")
    time.sleep(2)  # be polite to the API


# ============================================
# Climate indices
# NOTE for feature building: monthly ONI/DMI are published after the month ends,
# so when predicting on a given day use the PREVIOUS month's value to avoid
# using information that wasn't available yet. MJO RMM is available with ~1 day lag.
# ============================================
def download_mjo():
    url = "http://www.bom.gov.au/climate/mjo/graphics/rmm.74toRealtime.txt"
    text = get_with_retry(url).text
    rows = []
    for line in text.splitlines():
        parts = line.split()
        if len(parts) < 7 or not parts[0].isdigit():
            continue  # header lines
        y, m, d = int(parts[0]), int(parts[1]), int(parts[2])
        rmm1, rmm2, phase, amp = float(parts[3]), float(parts[4]), int(parts[5]), float(parts[6])
        if abs(rmm1) > 100 or abs(rmm2) > 100:  # missing values are stored as 1.E36 / 999
            continue
        rows.append((pd.Timestamp(y, m, d), rmm1, rmm2, phase, amp))
    df = pd.DataFrame(rows, columns=["DATE", "RMM1", "RMM2", "MJO_Phase", "MJO_Amplitude"])
    df = df[df["DATE"] >= pd.Timestamp(START_DATE)]
    df.to_csv(INDEX_DIR / "mjo_rmm_daily.csv", index=False)
    print(f"  saved MJO: {len(df)} days, last={df['DATE'].max().date()}")


def download_oni():
    url = "https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt"
    text = get_with_retry(url).text
    season_to_month = {s: i + 1 for i, s in enumerate(
        ["DJF", "JFM", "FMA", "MAM", "AMJ", "MJJ", "JJA", "JAS", "ASO", "SON", "OND", "NDJ"])}
    rows = []
    for line in text.splitlines():
        parts = line.split()
        if len(parts) == 4 and parts[0] in season_to_month:
            # ONI for a 3-month season is labelled by its centre month
            rows.append((int(parts[1]), season_to_month[parts[0]], float(parts[3])))
    df = pd.DataFrame(rows, columns=["Year", "Month", "ONI"])
    df = df[df["Year"] >= START_DATE.year]
    df.to_csv(INDEX_DIR / "enso_oni_monthly.csv", index=False)
    print(f"  saved ONI: {len(df)} months, last={df['Year'].iloc[-1]}-{df['Month'].iloc[-1]:02d}")


def download_dmi():
    url = "https://psl.noaa.gov/gcos_wgsp/Timeseries/Data/dmi.had.long.data"
    text = get_with_retry(url).text
    rows = []
    for line in text.splitlines():
        parts = line.split()
        if len(parts) != 13 or not parts[0].isdigit():
            continue  # header/footer lines
        year = int(parts[0])
        for month, val in enumerate(parts[1:], start=1):
            v = float(val)
            if v > -99:  # -9999 = missing
                rows.append((year, month, v))
    df = pd.DataFrame(rows, columns=["Year", "Month", "DMI"])
    df = df[df["Year"] >= START_DATE.year]
    df.to_csv(INDEX_DIR / "iod_dmi_monthly.csv", index=False)
    print(f"  saved DMI: {len(df)} months, last={df['Year'].iloc[-1]}-{df['Month'].iloc[-1]:02d}")


def download_extended():
    failures = []

    # Location metadata (useful later for distance/direction features)
    meta = [{"name": n, "role": "district", **c} for n, c in DISTRICTS.items()]
    meta += [{"name": n, "role": "regional", **c} for n, c in REGIONAL_POINTS.items()]
    pd.DataFrame(meta).to_csv(OUT_DIR / "locations.csv", index=False)

    print("=== Target districts ===")
    for name, coords in DISTRICTS.items():
        print(f"\n{name}")
        try:
            update_point(name, coords, DISTRICT_PARAMS, DISTRICT_DIR)
        except Exception as e:
            print(f"  FAILED: {e}")
            failures.append(name)

    print("\n=== Regional upstream points ===")
    for name, coords in REGIONAL_POINTS.items():
        print(f"\n{name} ({coords['direction']})")
        try:
            update_point(name, coords, REGIONAL_PARAMS, REGIONAL_DIR)
        except Exception as e:
            print(f"  FAILED: {e}")
            failures.append(name)

    print("\n=== Climate indices ===")
    for label, fn in [("MJO", download_mjo), ("ONI", download_oni), ("DMI", download_dmi)]:
        print(f"\n{label}")
        try:
            fn()
        except Exception as e:
            print(f"  FAILED: {e}")
            failures.append(label)

    print("\n" + ("All downloads completed successfully!" if not failures
                  else f"Finished with failures: {failures} -- re-run to retry them."))


if RUN_EXTENDED_DOWNLOAD:
    for d in (DISTRICT_DIR, REGIONAL_DIR, INDEX_DIR):
        d.mkdir(parents=True, exist_ok=True)
    download_extended()
else:
    print("skipped (RUN_EXTENDED_DOWNLOAD = False)")

## 4. Open-Meteo forecast (next 16 days)

Hourly values are aggregated to daily to match the NASA POWER columns. Only days 1-7 are reliable enough to use.
Open-Meteo has no dew point / cloud amount / 50 m wind here, so this covers 8 of the 11 LSTM variables.

In [ ]:
OPEN_METEO_URL = "https://api.open-meteo.com/v1/forecast"
FORECAST_DAYS = 16  # hard maximum the API allows
HOURLY = ["temperature_2m", "relative_humidity_2m", "precipitation",
          "surface_pressure", "windspeed_10m", "shortwave_radiation"]


def fetch_forecast(coords):
    params = {"latitude": coords["latitude"], "longitude": coords["longitude"],
              "hourly": ",".join(HOURLY), "forecast_days": FORECAST_DAYS, "timezone": "Asia/Kolkata"}
    response = requests.get(OPEN_METEO_URL, params=params, timeout=60)
    response.raise_for_status()
    hourly = pd.DataFrame(response.json()["hourly"])
    hourly["time"] = pd.to_datetime(hourly["time"])
    hourly["DATE"] = hourly["time"].dt.date
    return hourly.groupby("DATE").agg(
        Temperature_C=("temperature_2m", "mean"),
        Temperature_Max_C=("temperature_2m", "max"),
        Temperature_Min_C=("temperature_2m", "min"),
        Rainfall_mm=("precipitation", "sum"),
        Relative_Humidity_percent=("relative_humidity_2m", "mean"),
        Wind_Speed_mps=("windspeed_10m", "mean"),
        Surface_Pressure_kPa=("surface_pressure", "mean"),
        # W/m^2 hourly mean -> MJ/m^2/day, to match ALLSKY_SFC_SW_DWN's units
        Solar_Radiation=("shortwave_radiation", lambda s: s.mean() * 0.0036 * 24),
    ).reset_index()


if RUN_FORECAST_DOWNLOAD:
    out_dir = WEATHER_DIR / "forecast"
    out_dir.mkdir(parents=True, exist_ok=True)
    for name, coords in DISTRICTS.items():
        fc = fetch_forecast(coords)
        fc.to_csv(out_dir / f"{name.lower()}_weather_forecast.csv", index=False)
        print(f"{name}: {fc.DATE.min()} .. {fc.DATE.max()}")
else:
    print("skipped (RUN_FORECAST_DOWNLOAD = False)")

## 5. Cleaning

For each district in `TRAINING_VERSION`:
- drop trailing days where every value is missing (NASA POWER hasn't processed them yet)
- **cloud amount** gaps: fill with a smoothed day-of-year climatology computed from the training years only (no leakage)
- every other column: time-based interpolation

Output: `data/weather/clean/<district>_weather.csv`, the input to `forecasting/lstm.ipynb`.

In [ ]:
CLIM_COL = "Cloud_Amount_percent"


def load_raw(csv_path):
    raw = pd.read_csv(csv_path, parse_dates=["DATE"]).sort_values("DATE").reset_index(drop=True)
    features = [c for c in raw.columns if c != "DATE"]
    df = raw.copy()
    all_nan = df[features].isna().all(axis=1)
    if all_nan.any():
        df = df.loc[:df.index[~all_nan].max()].reset_index(drop=True)
    return df.set_index("DATE"), features


def fill_missing(df, features, train_end):
    df = df.copy()
    train_slice = df.loc[:train_end, CLIM_COL]
    clim = train_slice.groupby(train_slice.index.dayofyear).mean()
    clim_ext = pd.concat([clim, clim, clim])
    clim_smooth = clim_ext.rolling(7, center=True, min_periods=1).mean().iloc[len(clim):2 * len(clim)]
    clim_smooth.index = clim.index
    fill_values = df.index.dayofyear.map(clim_smooth)
    mask = df[CLIM_COL].isna()
    df.loc[mask, CLIM_COL] = fill_values[mask.values]

    other_cols = [c for c in features if c != CLIM_COL]
    df[other_cols] = df[other_cols].interpolate(method="time", limit_direction="both")
    assert df.isna().sum().sum() == 0, "NaNs remain after cleaning"
    return df


if RUN_CLEANING:
    clean_dir = WEATHER_DIR / "clean"
    clean_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for name in DISTRICTS:
        df, features = load_raw(WEATHER_DIR / TRAINING_VERSION / f"{name.lower()}_weather.csv")
        n_missing = int(df.isna().sum().sum())
        df = fill_missing(df, features, TRAIN_END)
        df.reset_index().to_csv(clean_dir / f"{name.lower()}_weather.csv", index=False)
        rows.append({"district": name, "days": len(df), "first": df.index.min().date(),
                     "last": df.index.max().date(), "values_filled": n_missing})
    display(pd.DataFrame(rows).set_index("district"))